# 6th attempt - RCNN - 5

In [1]:
# 1️⃣ ייבוא המודול והפונקציות
import functions         # ייבוא רגיל למודול
from functions import *  # ייבוא כל הפונקציות ל־namespace

# 2️⃣ טעינה מחדש
import importlib
importlib.reload(functions)  # מוודא שהמודול מעודכן

# 3️⃣ הרצת הפונקציות שוב ל־namespace
from functions import *  # שוב מייבא את כל הפונקציות אחרי ה־reload

In [2]:
import numpy as np
import pandas as pd
from functions import *
from read_from_file_df import *
import pickle
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [3]:
SIZE = 5
AMOUNT_BOARDS = 10000
AMOUNT_MOVES = 100
NUM_DICT = 1
gen = 2
gen_data = gen-1

In [4]:
reverse_df_sort = load_reverse_df(SIZE, AMOUNT_BOARDS, gen)
X_train, X_val, X_test, y_train, y_val, y_test = prepare_reverse_dataset(reverse_df_sort, SIZE, gen, target_pixel_index=0, test_size=0.1, val_size=0.1, random_state=365)
X_train_array, X_val_array, X_test_array, y_train_array, y_val_array, y_test_array = to_numpy_4d(X_train, X_val, X_test, y_train, y_val, y_test, SIZE, gen)

print("len df:", len(reverse_df_sort))
print("len train: ", len(X_train))
print("len val: ",len(X_val))
print("len test: ",len(X_test))

len df: 29760
len train:  24105
len val:  2679
len test:  2976


In [5]:
X_train.shape

(24105, 25)

In [6]:
model, history = build_and_train_rcnn(gen, X_train_array, y_train_array, SIZE, 32, 3,active = "softmax")

X_train shape: (24104, 1, 5, 5, 1)
y_train shape: (24104, 2)


c:\Users\דרור\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv_lstm2d (ConvLSTM2D)        │ (None, 1, 5, 5, 32)    │        38,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1, 5, 5, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_lstm2d_1 (ConvLSTM2D)      │ (None, 5, 5, 64)       │       221,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 5, 5, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       102,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 362,562 (1.38 MB)

 Trainable params: 362,370 (1.38 MB)

 Non-trainable params: 192 (768.00 B)

Epoch 1/3
603/603 ━━━━━━━━━━━━━━━━━━━━ 53s 66ms/step - accuracy: 0.6692 - loss: 0.6236 - val_accuracy: 0.6652 - val_loss: 0.6478
Epoch 2/3
603/603 ━━━━━━━━━━━━━━━━━━━━ 33s 51ms/step - accuracy: 0.7010 - loss: 0.5778 - val_accuracy: 0.6837 - val_loss: 0.5941
Epoch 3/3
603/603 ━━━━━━━━━━━━━━━━━━━━ 44s 54ms/step - accuracy: 0.7104 - loss: 0.5607 - val_accuracy: 0.6914 - val_loss: 0.5865


In [7]:
evaluate_model(model, X_test_array, y_test_array, size=SIZE, gen=gen)

93/93 ━━━━━━━━━━━━━━━━━━━━ 9s 44ms/step

===== Evaluation Results =====
┌──────────────┬────────────┬────────────┐
│              │ Pred=Alive │ Pred=Dead  │
├──────────────┼────────────┼────────────┤
│ True=Alive   │        376 │        645 │
│ True=Dead    │        299 │       1655 │
└──────────────┴────────────┴────────────┘

--- Performance Metrics ---
Accuracy    : 0.683
Precision   : 0.557
Recall      : 0.368
F1-score    : 0.443


In [9]:
import matplotlib.pyplot as plt
import numpy as np

def plot_results(y_test, y_pred):
    """
    מצייר פיזור בין הפלט האמיתי לפלט החזוי.
    """
    y_test = y_test.flatten()
    y_pred = y_pred.flatten()

    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, y_pred, alpha=0.3)
    
    # קו אידיאלי (תחזית = אמת)
    plt.plot([0, 1], [0, 1], linestyle='--')

    plt.xlabel("True Value (y_test)")
    plt.ylabel("Predicted Value (y_pred)")
    plt.title("Prediction Scatter Plot")
    plt.grid(True)
    plt.show()


In [15]:
amount_features = len(reverse_df_sort.columns) - SIZE*SIZE #the previous boards
features = reverse_df_sort.iloc[:, :amount_features]
for i in range(SIZE*SIZE): # to any pixel in the expected board
    X_train, X_val, X_test, y_train, y_val, y_test = prepare_reverse_dataset(reverse_df_sort, SIZE, gen, target_pixel_index=i, test_size=0.1, val_size=0.1, random_state=365)
    X_train_array, X_val_array, X_test_array, y_train_array, y_val_array, y_test_array = to_numpy_4d(X_train, X_val, X_test, y_train, y_val, y_test, SIZE, gen)
    model = build_and_train_rcnn(gen, X_train_array, y_train_array, SIZE, 32, 3, active="softmax")
    name_file = f"{PATH_MODELS}\\model6_RCNN_softmax\\{SIZE}\\size_{SIZE}_pixel_{str(i+1)}.pkl"
    with open(name_file, 'wb') as file:
        pickle.dump(model, file)
    print(i)

X_train shape: (24104, 1, 5, 5, 1)
y_train shape: (24104, 2)


c:\Users\דרור\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv_lstm2d_2 (ConvLSTM2D)      │ (None, 1, 5, 5, 32)    │        38,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 1, 5, 5, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_lstm2d_3 (ConvLSTM2D)      │ (None, 5, 5, 64)       │       221,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 5, 5, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │       102,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 362,562 (1.38 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv_lstm2d_50 (ConvLSTM2D)     │ (None, 1, 5, 5, 32)    │        38,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_50          │ (None, 1, 5, 5, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_lstm2d_51 (ConvLSTM2D)     │ (None, 5, 5, 64)       │       221,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_51          │ (None, 5, 5, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_25 (Flatten)            │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_50 (Dense)                │ (None, 64)             │       102,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_51 (Dense)                │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 362,562 (1.38 MB)

 Trainable params: 362,370 (1.38 MB)

 Non-trainable params: 192 (768.00 B)

Epoch 1/3
603/603 ━━━━━━━━━━━━━━━━━━━━ 77s 61ms/step - accuracy: 0.6699 - loss: 0.6246 - val_accuracy: 0.6573 - val_loss: 0.6292
Epoch 2/3
603/603 ━━━━━━━━━━━━━━━━━━━━ 36s 59ms/step - accuracy: 0.6964 - loss: 0.5839 - val_accuracy: 0.6945 - val_loss: 0.5887
Epoch 3/3
603/603 ━━━━━━━━━━━━━━━━━━━━ 40s 57ms/step - accuracy: 0.7127 - loss: 0.5653 - val_accuracy: 0.7042 - val_loss: 0.5821
24
